# Interactive all-pass ring resonator

This notebook accompanies the **Photonic Devices 2026** course.

It explores the spectral response of an **all-pass ring resonator**. The round-trip propagation factor is

$$
p(\lambda)=a\,e^{-j\phi(\lambda)}.
$$

The round-trip phase is

$$
\phi(\lambda)=\beta(\lambda)L,
\qquad
\beta(\lambda)=\frac{2\pi n_{\rm eff}(\lambda)}{\lambda}.
$$

The through-port transfer function is

$$
H_{\rm th}(\lambda)=\frac{t-p(\lambda)}{1-t\,p(\lambda)}.
$$

The power transmission is

$$
T(\lambda)=|H_{\rm th}(\lambda)|^2.
$$

The effective index is linearized around the reference wavelength $\lambda_0$:

$$
n_{\rm eff}(\lambda)
\simeq
n_{\rm eff,0}
+
\left.\frac{d n_{\rm eff}}{d\lambda}\right|_{\lambda_0}
(\lambda-\lambda_0).
$$

Using

$$
n_g=n_{\rm eff}-\lambda\frac{d n_{\rm eff}}{d\lambda},
$$

the slope at $\lambda_0$ is

$$
\left.\frac{d n_{\rm eff}}{d\lambda}\right|_{\lambda_0}
=
\frac{n_{\rm eff,0}-n_g}{\lambda_0}.
$$

The interactive controls allow you to change $t$, $L$, propagation loss, $n_{\rm eff,0}$, and $n_g$.

The reference wavelength is fixed at $\lambda_0=1.55\,\mu{\rm m}$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from matplotlib.ticker import ScalarFormatter

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

plt.rcParams["figure.figsize"] = (8.2, 4.5)

c0 = 299792458.0      # speed of light [m/s]
lambda0 = 1.55e-6     # reference wavelength [m]


## 1. Physical model

The field self-coupling coefficient of the bus-ring coupler is $t$. For an ideal lossless coupler,

$$
t^2+\kappa^2=1.
$$

The propagation loss is specified as a **power attenuation coefficient in dB/cm**. If the ring round-trip length is $L$, the corresponding field attenuation after one round trip is

$$
a=10^{-\alpha_{\rm dB/cm}L_{\rm cm}/20}.
$$

At resonance,

$$
\phi=2\pi m,
$$

and

$$
H_{\rm th,res}=\frac{t-a}{1-ta}.
$$

The ring is undercoupled for $t>a$, critically coupled for $t=a$, and overcoupled for $t<a$.

The free spectral range is approximately

$$
{\rm FSR}_f\simeq\frac{c}{n_gL},
$$

or, in wavelength,

$$
{\rm FSR}_\lambda\simeq\frac{\lambda_r^2}{n_gL}.
$$

For a sufficiently narrow resonance,

$$
\Delta\lambda_{\rm FWHM}
\simeq
{\rm FSR}_\lambda
\frac{1-ta}{\pi\sqrt{ta}},
$$

and

$$
Q=\frac{\lambda_r}{\Delta\lambda_{\rm FWHM}}.
$$


In [ ]:
def linear_neff(lambda_m, neff0, ng):
    """First-order model of n_eff(lambda). All wavelengths are in metres."""
    dneff_dlambda = (neff0 - ng) / lambda0
    return neff0 + dneff_dlambda * (lambda_m - lambda0)


def resonance_wavelength(m, L_m, neff0, ng):
    """Resonance wavelength [m] for the linearized effective-index model."""
    dneff_dlambda = (neff0 - ng) / lambda0

    # n_eff(lambda) = ng + (dn_eff/dlambda) lambda
    # resonance: n_eff(lambda) L = m lambda
    denominator = m - L_m * dneff_dlambda

    if denominator <= 0:
        return np.nan

    return ng * L_m / denominator


def ring_quantities(t, L_um, loss_dB_cm, neff0, ng):
    """Calculate the main quantities of the all-pass ring."""
    if not (0.0 < t < 1.0):
        raise ValueError("t must satisfy 0 < t < 1.")
    if L_um <= 0:
        raise ValueError("L must be positive.")
    if loss_dB_cm < 0:
        raise ValueError("Propagation loss cannot be negative.")
    if neff0 <= 0 or ng <= 0:
        raise ValueError("Refractive indices must be positive.")

    L_m = L_um * 1e-6
    L_cm = L_m * 100.0

    a = 10.0**(-loss_dB_cm * L_cm / 20.0)
    kappa = np.sqrt(1.0 - t**2)

    m0 = int(np.round(neff0 * L_m / lambda0))
    lambda_r = resonance_wavelength(m0, L_m, neff0, ng)

    FSR_f = c0 / (ng * L_m)
    FSR_lambda = lambda_r**2 / (ng * L_m)

    rt = t * a
    FWHM_lambda = (
        FSR_lambda
        * (1.0 - rt)
        / (np.pi * np.sqrt(rt))
    )

    finesse = FSR_lambda / FWHM_lambda
    Q = lambda_r / FWHM_lambda

    T_res = ((t - a) / (1.0 - t*a))**2

    if np.isclose(t, a, atol=2e-4, rtol=0.0):
        regime = "critical coupling"
    elif t > a:
        regime = "undercoupled"
    else:
        regime = "overcoupled"

    return {
        "L_m": L_m,
        "a": a,
        "kappa": kappa,
        "m0": m0,
        "lambda_r": lambda_r,
        "FSR_f": FSR_f,
        "FSR_lambda": FSR_lambda,
        "FWHM_lambda": FWHM_lambda,
        "finesse": finesse,
        "Q": Q,
        "T_res": T_res,
        "regime": regime,
    }


def ring_spectrum(t, L_um, loss_dB_cm, neff0, ng, n_fsr=3.2, npts=12000):
    """Calculate power and phase over several FSRs around the selected resonance."""
    q = ring_quantities(t, L_um, loss_dB_cm, neff0, ng)

    # lambda_r and FSR_lambda are both in metres.
    half_span = 40e-9#n_fsr * q["FSR_lambda"]
    lambda_min = q["lambda_r"] - half_span
    lambda_max = q["lambda_r"] + half_span

    lambda_m = np.linspace(lambda_min, lambda_max, npts)

    neff = linear_neff(lambda_m, neff0, ng)
    beta = 2.0 * np.pi * neff / lambda_m
    phi = beta * q["L_m"]

    p = q["a"] * np.exp(-1j * phi)
    H = (t - p) / (1.0 - t*p)

    T = np.abs(H)**2
    phase = np.unwrap(np.angle(H))

    return lambda_m, T, phase, q


## 2. Interactive all-pass ring explorer

The plotting range follows the FSR automatically, so several consecutive resonances remain visible when the ring parameters are changed.

All wavelength calculations are performed internally in SI units. Conversion to nanometres is made only when displaying results and plots.


In [ ]:
def update_ring(t, L_um, loss_dB_cm, neff0, ng):
    try:
        lambda_m, T, phase, q = ring_spectrum(
            t=t,
            L_um=L_um,
            loss_dB_cm=loss_dB_cm,
            neff0=neff0,
            ng=ng,
        )
    except ValueError as exc:
        print(exc)
        return

    lambda_nm = lambda_m * 1e9
    lambda_r_nm = q["lambda_r"] * 1e9
    FSR_nm = q["FSR_lambda"] * 1e9
    FWHM_nm = q["FWHM_lambda"] * 1e9

    resonance_nm = []
    for m in range(q["m0"] - 8, q["m0"] + 9):
        lr = resonance_wavelength(m, q["L_m"], neff0, ng)
        if np.isfinite(lr) and lambda_m[0] <= lr <= lambda_m[-1]:
            resonance_nm.append(lr * 1e9)

    print("ALL-PASS RING RESONATOR")
    print("-" * 52)
    print(f"Reference wavelength        = {lambda0*1e9:.1f} nm")
    print(f"Round-trip length L         = {L_um:.3f} um")
    print(f"Equivalent radius R         = {L_um/(2*np.pi):.3f} um")
    print(f"n_eff(lambda0)              = {neff0:.4f}")
    print(f"n_g                         = {ng:.4f}")
    print(f"Propagation loss            = {loss_dB_cm:.3f} dB/cm")
    print()
    print(f"Round-trip field factor a   = {q['a']:.6f}")
    print(f"Self-coupling t             = {t:.6f}")
    print(f"Cross-coupling kappa        = {q['kappa']:.6f}")
    print(f"Coupling regime             = {q['regime']}")
    print(f"Critical-coupling value     = t = a = {q['a']:.6f}")
    print()
    print(f"Nearest resonance           = {lambda_r_nm:.4f} nm")
    print(f"Resonance order m           = {q['m0']}")
    print(f"FSR                         = {q['FSR_f']/1e9:.3f} GHz")
    print(f"FSR                         = {FSR_nm:.4f} nm")
    print(f"FWHM                        = {FWHM_nm:.5f} nm")
    print(f"Finesse                     = {q['finesse']:.2f}")
    print(f"Q                           = {q['Q']:.3e}")
    print(f"Transmission at resonance   = {q['T_res']:.6f}")
    print(f"Displayed wavelength range  = {lambda_nm[0]:.2f} - {lambda_nm[-1]:.2f} nm")

    fig, ax = plt.subplots(figsize=(8.6, 4.5))
    ax.plot(lambda_nm, T, linewidth=1.6)

    for lr_nm in resonance_nm:
        ax.axvline(lr_nm, linestyle=":", linewidth=0.8, alpha=0.55)

    ax.axvline(
        lambda_r_nm,
        linestyle="--",
        linewidth=1.1,
        label="selected resonance",
    )

    ax.set_xlabel("Wavelength [nm]")
    ax.set_ylabel("Power transmission")
    ax.set_title(f"All-pass ring: {q['regime']}")
    ax.set_ylim(0.0, 1.05)
    ax.set_xlim(lambda_nm[0], lambda_nm[-1])
    ax.grid(True, alpha=0.3)
    ax.legend()

    formatter = ScalarFormatter(useOffset=False)
    formatter.set_scientific(False)
    ax.xaxis.set_major_formatter(formatter)

    plt.show()

    fig, ax = plt.subplots(figsize=(8.6, 4.2))
    ax.plot(lambda_nm, phase, linewidth=1.6)
    ax.axvline(lambda_r_nm, linestyle="--", linewidth=1.1)
    ax.set_xlabel("Wavelength [nm]")
    ax.set_ylabel("Phase of H [rad]")
    ax.set_title("Through-port phase response")
    ax.set_xlim(lambda_nm[0], lambda_nm[-1])
    ax.grid(True, alpha=0.3)

    formatter = ScalarFormatter(useOffset=False)
    formatter.set_scientific(False)
    ax.xaxis.set_major_formatter(formatter)

    plt.show()


slider_style = {"description_width": "125px"}
slider_layout = widgets.Layout(width="390px")

t_slider = widgets.FloatSlider(
    value=0.9800,
    min=0.0005,
    max=0.9995,
    step=0.0005,
    description="t:",
    readout_format=".4f",
    continuous_update=False,
    style=slider_style,
    layout=slider_layout,
)

L_slider = widgets.FloatSlider(
    value=2*np.pi*10.0,
    min=20.0,
    max=300.0,
    step=0.2,
    description="L (um):",
    readout_format=".2f",
    continuous_update=False,
    style=slider_style,
    layout=slider_layout,
)

loss_slider = widgets.FloatSlider(
    value=27.93,
    min=0.0,
    max=100.0,
    step=0.05,
    description="loss (dB/cm):",
    readout_format=".2f",
    continuous_update=False,
    style=slider_style,
    layout=slider_layout,
)

neff_slider = widgets.FloatSlider(
    value=2.45,
    min=1.20,
    max=3.50,
    step=0.01,
    description="n_eff(lambda0):",
    readout_format=".2f",
    continuous_update=False,
    style=slider_style,
    layout=slider_layout,
)

ng_slider = widgets.FloatSlider(
    value=4.00,
    min=1.20,
    max=6.00,
    step=0.01,
    description="n_g:",
    readout_format=".2f",
    continuous_update=False,
    style=slider_style,
    layout=slider_layout,
)

controls = widgets.VBox([
    widgets.HBox([t_slider, L_slider]),
    widgets.HBox([loss_slider, neff_slider]),
    widgets.HBox([ng_slider]),
])

interactive_output = widgets.interactive_output(
    update_ring,
    {
        "t": t_slider,
        "L_um": L_slider,
        "loss_dB_cm": loss_slider,
        "neff0": neff_slider,
        "ng": ng_slider,
    },
)

display(controls, interactive_output)


## 3. Suggested exploration

Try to predict each trend before moving the sliders.

1. Increase $L$ and verify that the FSR decreases.
2. Change $n_g$ and observe the change in resonance spacing.
3. Change $n_{\rm eff,0}$ and observe the shift of the resonance wavelengths.
4. Keep the propagation loss fixed and vary $t$ across the critical-coupling value $t=a$.
5. Compare undercoupled, critically coupled, and overcoupled responses.
6. Increase the propagation loss and observe the changes in linewidth and $Q$.
7. Compare the numerical spectrum with the analytical FSR and FWHM expressions.
